# cbx-healthcare-speech-to-text-0.6B — English medical consultation ASR on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/cbx-healthcare-speech-to-text-0.6B-medical-asr.ipynb)

[`CobuildX-AI/cbx-healthcare-speech-to-text-0.6B`](https://huggingface.co/CobuildX-AI/cbx-healthcare-speech-to-text-0.6B) is a **PEFT LoRA adapter** on [`Qwen/Qwen3-ASR-0.6B`](https://huggingface.co/Qwen/Qwen3-ASR-0.6B) for **English doctor–patient consultation** transcription. [Apache-2.0](https://huggingface.co/CobuildX-AI/cbx-healthcare-speech-to-text-0.6B), public (not gated). The card says to pin revision **v0.1**. Repo files: `adapter_config.json`, `adapter_model.safetensors` (~40 MB), `stt.py` (`SpeechToText`), `requirements.txt`, `example.py`. The base checkpoint `Qwen/Qwen3-ASR-0.6B` `model.safetensors` is about **1.9 GB**.

**Why this model.** The card's `SpeechToText` wrapper is the documented API (`stt.transcribe(path | (samples, sr) | 16 kHz array, context="expected words")` and `stt.transcribe_batch([...])`). On [PriMock57](https://github.com/babylonhealth/primock57) mock UK GP consultations (4,476 utterances, never used in training) the card reports **word error rate 6.01%** vs **6.00%** for the base model, **condition names wrong 5.4%** vs **16.1%**, and **medicine names wrong 33.9%** vs **38.8%**. Those are the card's numbers, not a T4 measurement from this notebook.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/cbx-healthcare-speech-to-text-0.6B-medical-asr.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0, the same as the base model. Training data on the card: [Synth-DoPaCo](https://huggingface.co/datasets/Play-Your-Part/Synth-DoPaCo) (CC-BY-4.0) and the [Fareez et al. 2022 OSCE dataset](https://doi.org/10.6084/m9.figshare.16550013) (CC0).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~16 GB of VRAM and **no native bfloat16**. `SpeechToText` merges the adapter in fp32, then runs **float16 on CUDA** (fp32 on CPU). The install cell uninstalls Colab's `torchao` 0.10.0 (peft 0.21.2 raises on that version), pins `huggingface_hub>=0.34,<1`, and **restarts the session**. The card reports on an NVIDIA T4: **1 minute of speech in about 2.4 s** (fp16, batches of 16 clips). This notebook uses the card's default `batch_size=8` on short clips, so wall time here is not that 2.4 s figure.

> **Human review — not a medical device.** The card: for clinical records, have transcripts reviewed before they are used. English only. Training speech is US-based; accuracy on other accents has not been fully measured. For other speech, such as lectures or single short sentences, the card says the base model is currently more accurate.

**Not in this notebook:** training, vLLM serving, or any real clinical workflow.

**Links:** [model card](https://huggingface.co/CobuildX-AI/cbx-healthcare-speech-to-text-0.6B) · [base Qwen3-ASR-0.6B](https://huggingface.co/Qwen/Qwen3-ASR-0.6B) · [PriMock57](https://github.com/babylonhealth/primock57) · [announcement](https://x.com/sairaamv/status/2122704454306922853) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/cbx-healthcare-speech-to-text-0.6B-medical-asr.md)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; SpeechToText will run float16 on CUDA.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Download revision v0.1 and install (keep Colab's torch)

The [model card](https://huggingface.co/CobuildX-AI/cbx-healthcare-speech-to-text-0.6B) install is:

```bash
hf download CobuildX-AI/cbx-healthcare-speech-to-text-0.6B --revision v0.1 --local-dir cbx-healthcare-speech-to-text-0.6B
cd cbx-healthcare-speech-to-text-0.6B
pip install -r requirements.txt
```

`requirements.txt` (v0.1) pins `torch>=2.4`, `qwen-asr==0.0.6`, `transformers==4.57.6`, `peft==0.21.2`, `librosa`. **Do not reinstall torch** — Colab already ships a CUDA wheel; forcing a reinstall often drops GPU support. `qwen-asr==0.0.6` also declares CLI extras (`gradio`, `flask`, `sox`) that this notebook does not use; `sox` often fails on Colab without a system libsox. SpeechToText only needs the transformers backend. `qwen_asr` imports `nagisa` through `Qwen3ForcedAligner` even for plain ASR, so nagisa stays.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch.

**Colab-only pins.** `peft==0.21.2` calls `is_torchao_available()` on import and **raises** if Colab's preinstalled `torchao==0.10.0` is present (it wants `>=0.16`). This notebook does not use torchao, so the install cell uninstalls it rather than upgrading it on top of Colab's `torch 2.14`. `transformers==4.57.6` requires `huggingface_hub>=0.34,<1` — do not `-U huggingface_hub` to 2.x (that also trips Colab `HF_TOKEN` secret popups).

**Do not import `stt` in this cell.** Colab has already imported its preinstalled transformers. This cell restarts the session for you so 4.57.6 is what gets imported. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

ADAPTER_REPO = "CobuildX-AI/cbx-healthcare-speech-to-text-0.6B"
REVISION = "v0.1"
INSTALL_TAG = "v0.1-tf4.57.6-no-torchao-hublt1"
ADAPTER_DIR = (
    Path("/content/cbx-healthcare-speech-to-text-0.6B")
    if Path("/content").is_dir()
    else Path("cbx-healthcare-speech-to-text-0.6B")
)
MARKER = (
    Path("/content/.cbx_stt_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".cbx_stt_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# transformers 4.57.6 requires huggingface_hub>=0.34,<1. Do not -U to 2.x.
get_ipython().run_line_magic("pip", 'install -q "huggingface_hub>=0.34.0,<1"')

# Public weights: skip Colab userdata.get("HF_TOKEN") popups. Hub still calls
# get_token() even with HF_HUB_DISABLE_IMPLICIT_TOKEN set; token=False does not.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
print("snapshot_download", ADAPTER_REPO, "revision", REVISION, "->", ADAPTER_DIR)
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=ADAPTER_REPO,
    revision=REVISION,
    local_dir=str(ADAPTER_DIR),
    token=False,
)

req_path = ADAPTER_DIR / "requirements.txt"
print()
print("=== requirements.txt (v0.1) ===")
print(req_path.read_text())
print("=== adapter files ===")
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:28s} {p.stat().st_size / 1024**2:.2f} MB")

# Card pins. Skip torch. Skip qwen-asr extras that are CLI-only (gradio/flask/sox).
# qwen-asr==0.0.6 needs transformers==4.57.6 exactly (its bundled Qwen3-ASR code).
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers==4.57.6" "peft==0.21.2" "accelerate==1.12.0" '
    '"huggingface_hub>=0.34.0,<1" librosa soundfile "nagisa==0.2.11" "soynlp==0.0.493"',
)
get_ipython().run_line_magic("pip", 'install -q --no-deps "qwen-asr==0.0.6"')

# peft 0.21.2 raises ImportError on Colab's torchao 0.10.0. We do not use torchao.
get_ipython().run_line_magic("pip", "uninstall -y torchao")

print()
print("transformers", md.version("transformers"))
print("peft", md.version("peft"))
print("qwen-asr", md.version("qwen-asr"))
print("huggingface_hub", md.version("huggingface_hub"))
try:
    print("torchao", md.version("torchao"), "(should be gone)")
except md.PackageNotFoundError:
    print("torchao: not installed (good)")
print("ADAPTER_DIR", ADAPTER_DIR.resolve())

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so transformers==4.57.6 is imported and peft")
    print("does not see Colab's torchao 0.10.0.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you**. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers==4.57.6`, CUDA, device name, and free VRAM. Then define paths and a VRAM helper. T4 (`sm_75`) has no native bfloat16 — **float16** is what `SpeechToText` uses on CUDA.


In [ ]:
import importlib.util
import os
import sys
import time
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
import transformers

assert transformers.__version__ == "4.57.6", (
    f"transformers {transformers.__version__} is loaded; qwen-asr==0.0.6 / this adapter "
    "pin transformers==4.57.6. Run the install cell, then Runtime > Restart session."
)

if importlib.util.find_spec("torchao") is not None:
    get_ipython().run_line_magic("pip", "uninstall -y torchao")
    raise SystemExit(
        "Uninstalled Colab torchao (peft 0.21.2 raises on 0.10.0; we do not use it). "
        "Runtime > Restart session, then continue from this cell."
    )

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
ADAPTER_REPO = "CobuildX-AI/cbx-healthcare-speech-to-text-0.6B"
REVISION = "v0.1"
BASE_MODEL = "Qwen/Qwen3-ASR-0.6B"
ADAPTER_DIR = (
    Path("/content/cbx-healthcare-speech-to-text-0.6B")
    if Path("/content/cbx-healthcare-speech-to-text-0.6B").is_dir()
    else Path("cbx-healthcare-speech-to-text-0.6B")
)
assert (ADAPTER_DIR / "stt.py").is_file(), (
    f"stt.py not in {ADAPTER_DIR.resolve()}. Re-run the install cell, then restart."
)
sys.path.insert(0, str(ADAPTER_DIR.resolve()))

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("torchao: not installed (good)")
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free_b / 1024**3:.1f} GB of {total_b / 1024**3:.1f} GB")
print("ADAPTER_DIR", ADAPTER_DIR.resolve())
if major < 8:
    print("No native bfloat16 on this GPU — SpeechToText will use float16 on CUDA.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB | "
        f"free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load SpeechToText

The card: `from stt import SpeechToText; stt = SpeechToText()`. That loads `Qwen/Qwen3-ASR-0.6B` plus the adapter in this folder, and uses a GPU if one is available. First run downloads the **1.9 GB** base checkpoint.


In [ ]:
from stt import SpeechToText

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
stt = SpeechToText()
if torch.cuda.is_available():
    torch.cuda.synchronize()
print(f"loaded SpeechToText in {time.perf_counter() - t0:.1f}s")
print("device:", stt.device)
print("base:", getattr(stt.asr, "model", None).__class__.__name__)
print_vram("after adapter load")


## 4. Demo 1 — PriMock57 mock GP consultation (reference vs output)

The card's `example.py` downloads a short mock GP consultation from [PriMock57](https://github.com/babylonhealth/primock57) (`day1_consultation01_doctor`) and prints what was said next to what the model wrote. Same source here. Clips are 2–30 s doctor-channel intervals with no `<...>` uncertainty marks, matching `example.py`.

If the PriMock57 download fails, the cell falls back to a public Hugging Face speech sample, then to a generated 16 kHz tone so later cells still run. Fallback audio is **not** a medical consultation.


In [ ]:
import re
import urllib.request
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

SR = 16_000
SAMPLE_DIR = Path("cbx_stt_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}

PRIMOCK_WAV_URL = (
    "https://media.githubusercontent.com/media/babylonhealth/primock57/main"
    "/audio/day1_consultation01_doctor.wav"
)
PRIMOCK_GRID_URL = (
    "https://raw.githubusercontent.com/babylonhealth/primock57/main"
    "/transcripts/day1_consultation01_doctor.TextGrid"
)
HF_SPEECH_SAMPLE = "https://cdn-media.huggingface.co/speech_samples/sample1.flac"


def _urlretrieve(url, path, timeout=120):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as resp, open(path, "wb") as f:
        f.write(resp.read())
    return Path(path)


def parse_textgrid_intervals(y, grid_text):
    # Every nonempty doctor interval. example.py then keeps 2-30 s clips with no '<' tags.
    grid_text = grid_text.replace("\r\n", "\n").replace("\r", "\n")
    intervals = []
    for s, e, said in re.findall(
        r'xmin = ([\d.]+)\s*\n\s*xmax = ([\d.]+)\s*\n\s*text = "(.*?)"\s*\n',
        grid_text,
        flags=re.S,
    ):
        s, e, said = float(s), float(e), said.strip()
        if not said:
            continue
        intervals.append(
            {
                "audio": y[int(s * SR) : int(e * SR)],
                "reference": said,
                "start": s,
                "end": e,
            }
        )
    return intervals


def example_py_clips(intervals, max_clips=4):
    # Card example.py: 2–30 s, skip intervals with '<' (PriMock57 <UNSURE>/<UNIN/> marks).
    out = []
    for clip in intervals:
        dur = clip["end"] - clip["start"]
        if clip["reference"] and "<" not in clip["reference"] and 2 <= dur <= 30:
            out.append(clip)
        if len(out) >= max_clips:
            break
    return out


def write_clip(path, audio):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=-1)
    sf.write(path, audio, SR)
    return path, len(audio) / SR


clips = []
all_clips = []
source_label = None

try:
    wav_path = SAMPLE_DIR / "day1_consultation01_doctor.wav"
    grid_path = SAMPLE_DIR / "day1_consultation01_doctor.TextGrid"
    if not wav_path.exists():
        print("Downloading PriMock57 doctor-channel wav + TextGrid...")
        _urlretrieve(PRIMOCK_WAV_URL, wav_path)
        _urlretrieve(PRIMOCK_GRID_URL, grid_path)
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    all_clips = parse_textgrid_intervals(y, grid_path.read_text(encoding="utf-8"))
    clips = example_py_clips(all_clips)
    if not clips:
        raise RuntimeError("PriMock57 TextGrid produced no 2–30 s clips")
    source_label = "PriMock57 day1_consultation01_doctor"
    print(f"{source_label}: {len(y) / SR:.1f}s audio, {len(clips)} demo clips (example.py takes 4)")
except Exception as exc:
    print(f"PriMock57 download/parse failed: {exc!r}")
    try:
        fallback = SAMPLE_DIR / "hf-speech-sample1.flac"
        print("Fallback: Hugging Face speech_samples/sample1.flac")
        _urlretrieve(HF_SPEECH_SAMPLE, fallback)
        y, _ = librosa.load(fallback, sr=SR, mono=True)
        clips = [{"audio": y, "reference": None, "start": 0.0, "end": len(y) / SR}]
        all_clips = clips
        source_label = "Hugging Face speech_samples/sample1.flac (not medical speech)"
    except Exception as exc2:
        print(f"HF speech sample failed: {exc2!r}")
        print("Last-resort fallback: 3 s of 16 kHz silence so later cells still run.")
        y = np.zeros(SR * 3, dtype=np.float32)
        clips = [{"audio": y, "reference": None, "start": 0.0, "end": 3.0}]
        all_clips = clips
        source_label = "generated 3 s silence (not speech)"

print("source:", source_label)
batch = []
for i, clip in enumerate(clips, 1):
    path, seconds = write_clip(SAMPLE_DIR / f"demo1-{i:02d}.wav", clip["audio"])
    clip["path"] = path
    clip["seconds"] = seconds
    batch.append((clip["audio"], SR))
    print(f"\nclip {i}: {seconds:.2f}s  file={path.name}")
    if clip["reference"]:
        print("  reference:", clip["reference"])
    display(Audio(str(path)))

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
demo1_out = stt.transcribe_batch(batch)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
print()
print(f"transcribe_batch({len(batch)} clips) in {elapsed:.2f}s")
for i, (clip, out) in enumerate(zip(clips, demo1_out), 1):
    print(f"\n{i}. seconds={out['seconds']}  latency_ms={out['latency_ms']}")
    if clip["reference"]:
        print("   Said: ", clip["reference"])
    print("   Model:", out["text"])
print_vram("after PriMock57 / fallback clips")


## 5. Demo 2 — base Qwen3-ASR-0.6B vs this adapter

Same clip, two models. The adapter is already in VRAM. Loading the **0.6B base** a second time in float16 is about another 2 GB — cheap on a 16 GB T4. If that OOM's, the cell skips the comparison instead of crashing.

The card's PriMock57 table is the published comparison (WER ~tied; fewer condition/medicine mistakes with the adapter). This cell is a single-clip check, not that table.


In [ ]:
from qwen_asr import Qwen3ASRModel

compare_clip = clips[0]
compare_audio = (compare_clip["audio"], SR)
print("clip:", compare_clip["path"].name, f"({compare_clip['seconds']:.2f}s)")
if compare_clip["reference"]:
    print("reference:", compare_clip["reference"])
print("adapter:", demo1_out[0]["text"])

base_asr = None
try:
    torch.cuda.empty_cache()
    t0 = time.perf_counter()
    base_asr = Qwen3ASRModel.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16,
        device_map="cuda",
        max_inference_batch_size=8,
        max_new_tokens=256,
        token=False,
    )
    torch.cuda.synchronize()
    print(f"loaded base {BASE_MODEL} in {time.perf_counter() - t0:.1f}s")
    print_vram("after base load (adapter still in VRAM)")

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    base_out = base_asr.transcribe(
        audio=[compare_audio],
        language=["English"],
        context=[""],
    )[0]
    torch.cuda.synchronize()
    print(f"base transcribe {time.perf_counter() - t0:.2f}s")
    print("base:   ", base_out.text)
    print("adapter:", demo1_out[0]["text"])
except torch.cuda.OutOfMemoryError:
    print("CUDA OOM loading the base next to the adapter — skipping the side-by-side.")
    print("The card's PriMock57 table is the published comparison.")
    torch.cuda.empty_cache()
except Exception as exc:
    print(f"Base comparison skipped: {type(exc).__name__}: {exc}")
    torch.cuda.empty_cache()

print_vram("after base vs adapter")


## 6. Demo 3 — `context=` medicine-list hint

The card: `stt.transcribe("visit.wav", context="Clenil, salbutamol")` — optional words to expect, such as a clinic's medicine list (may help with rare names). The card lists **Clenil, Implanon and Dioralyte** as UK brand names that never appear in the training data, and says medicines are still wrong about one time in three.

If this session has the PriMock57 doctor channel, the cell prefers a clip whose reference mentions **Dioralyte** or **paracetamol** (both occur in `day1_consultation01_doctor`; some of those intervals are tagged `<UNIN/>` / `<UNSURE>` in the TextGrid, so they are skipped by `example.py` but still useful here). The context string is the card's example list plus those two names when present. On fallback audio the same API call still runs.


In [ ]:
CARD_CONTEXT = "Clenil, salbutamol"
EXTRA_FROM_PRIMOCK = ("Dioralyte", "paracetamol")

context_clip = None
for c in all_clips:
    ref = (c.get("reference") or "").lower()
    if any(name.lower() in ref for name in EXTRA_FROM_PRIMOCK):
        context_clip = c
        break
if context_clip is None:
    context_clip = clips[0]

if "path" not in context_clip:
    path, seconds = write_clip(SAMPLE_DIR / "demo3-context.wav", context_clip["audio"])
    context_clip["path"] = path
    context_clip["seconds"] = seconds

hint_parts = [CARD_CONTEXT]
if context_clip.get("reference"):
    present = [n for n in EXTRA_FROM_PRIMOCK if n.lower() in context_clip["reference"].lower()]
    if present:
        hint_parts.append(", ".join(present))
context_hint = ", ".join(hint_parts)

print("clip:", context_clip["path"].name, f"({context_clip['seconds']:.2f}s)")
if context_clip.get("reference"):
    print("reference:", context_clip["reference"])
print("context:", context_hint)
display(Audio(str(context_clip["path"])))

torch.cuda.synchronize()
t0 = time.perf_counter()
plain = stt.transcribe(str(context_clip["path"]))
with_ctx = stt.transcribe(str(context_clip["path"]), context=context_hint)
torch.cuda.synchronize()
print(f"two transcribe() calls in {time.perf_counter() - t0:.2f}s")
print()
print("without context:", plain["text"])
print("with context:   ", with_ctx["text"])
print("(A one-clip demo of the card's context= argument, not a medicine-name benchmark.)")
print_vram("after context demo")


## 7. Optional — transcribe your own audio

Set `UPLOAD_OWN_AUDIO = True` and re-run this cell to pick a wav / flac / mp3 / ogg from your computer. Default is `False` so **Run all** does not block on the Colab file picker. Do not upload real patient recordings to Colab.


In [ ]:
UPLOAD_OWN_AUDIO = False  # set True and re-run this cell to pick a file

if not UPLOAD_OWN_AUDIO:
    print("Skipped (UPLOAD_OWN_AUDIO = False). Set True to upload a clip from your computer.")
else:
    try:
        from google.colab import files
    except ImportError:
        files = None

    uploaded = None
    if files is not None:
        uploaded = files.upload()
    if not uploaded:
        print("No file uploaded. Put a wav/flac/mp3/ogg in the working directory and set AUDIO_PATH.")
    else:
        name = next(iter(uploaded))
        audio_path = Path(name)
        print("uploaded", audio_path, f"({audio_path.stat().st_size / 1024:.1f} KB)")
        display(Audio(str(audio_path)))
        out = stt.transcribe(str(audio_path))
        print("seconds:", out["seconds"], "| latency_ms:", out["latency_ms"])
        print("text:", out["text"])
        print_vram("after uploaded clip")


## 8. Caveats and cleanup

**What this notebook is for.** English doctor–patient consultation ASR on a free T4 with the card's `SpeechToText` API (PriMock57 mock clips, a cheap base-vs-adapter check, and `context=` as a medicine-list hint).

**What it is not.** A clinical product. The card: have transcripts reviewed before they are used. English only. Training speech is US-based. For lectures or single short sentences, the card says the base model is currently more accurate. Medicines are still wrong about one time in three on the card's PriMock57 count.

**Related notebooks:** [Fluid Speech Pro](Fluid-Speech-Pro-multilingual-asr.ipynb) (25 European languages), [Nemotron-Labs Unified ASR 0.6B](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb), [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb) (keyword-biased ASR + translation), [oruk-loop-153M](oruk-loop-153M-looped-asr.ipynb) (English ASR), [Whisper](AIBackends-Whisper-audio-transcription.ipynb).

Free the GPU before you open another notebook:


In [ ]:
import gc

print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del stt
if "base_asr" in globals():
    del base_asr
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
